In [1]:
#KAGGLE FOR DATA SET




import numpy as np            # For numerical operations and handling arrays
import pandas as pd           # For data manipulation and creating DataFrames
import seaborn as sns         # High-level interface for drawing statistical graphics
import matplotlib.pyplot as plt
import warnings               # To handle library-specific warnings
warnings.filterwarnings("ignore") # Suppress warnings to keep the output clean and readable

# --- Scikit-Learn Modules ---

# Splitting data into Training (to learn) and Testing (to validate) sets
from sklearn.model_selection import train_test_split

# Tool to apply different preprocessing steps to different columns simultaneously
from sklearn.compose import ColumnTransformer

# OneHotEncoder: Converts categories (text) to numbers (0s and 1s)
# StandardScaler: Scales numbers so they have a mean of 0 and variance of 1
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Pipeline: A 'container' that chains together preprocessing and the model
from sklearn.pipeline import Pipeline

# The algorithm we are using for prediction (Linear Regression)
from sklearn.ensemble import RandomForestRegressor

# Evaluation Metrics: To check how 'wrong' or 'right' our model is
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
#load data
# Load the dataset from a CSV file into a pandas DataFrame named 'df'
df = pd.read_csv("House_Rent_Dataset.csv")

In [3]:
#Features and target
X = df.drop("Rent", axis=1)
Y = df["Rent"]

In [7]:
#log transform target (VERY IMPORTANT)
# Apply natural log transformation: y = log(1 + y)
# This compresses extremely high rent values (outliers) 
# and helps the model achieve a more "Normal" distribution.
Y = np.log1p(Y)

In [8]:
#Select useful columns

# List the columns that contain numbers
numeric_data = ['BHK', 'Size', 'Bathroom']

# List the columns that contain text/categories
category_data = ['Area Type', 'Area Locality', 'City', 'Furnishing Status', 'Floor']

# Reorganize X to only include these specific features in the given order
X = X[numeric_data + category_data]

In [9]:
Y

0       2.323411
1       2.389087
2       2.374071
3       2.323411
4       2.294834
          ...   
4741    2.362350
4742    2.422595
4743    2.439136
4744    2.460822
4745    2.362350
Name: Rent, Length: 4746, dtype: float64

In [12]:
#Train-Test-Split
# Splitting the data: 75% for training the model, 25% for testing its accuracy
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.25, random_state=42
)

In [13]:
X_train

,BHK,Size,Bathroom,Area Type,Area Locality,City,Furnishing Status,Floor
1196,2,1300,2,Carpet Area,Khar West,Mumbai,Furnished,Upper Basement out of 10
2215,2,630,2,Super Area,Whitefield,Bangalore,Furnished,3 out of 5
1598,1,600,1,Super Area,Madanayakahalli,Bangalore,Unfurnished,1 out of 4
1532,2,1045,2,Carpet Area,Maruthi Sevanagar,Bangalore,Semi-Furnished,1 out of 2
1501,2,600,1,Super Area,Talaghattapura,Bangalore,Furnished,3 out of 4
...,...,...,...,...,...,...,...,...
4426,3,1500,3,Carpet Area,"Ayodhya Nagar, Quthbullapur",Hyderabad,Semi-Furnished,1 out of 2
466,3,1200,2,Super Area,Bansdroni,Kolkata,Unfurnished,3 out of 3
3092,2,800,2,Carpet Area,Vadapalani,Chennai,Semi-Furnished,13 out of 17
3772,3,3500,3,Carpet Area,T Nagar,Chennai,Semi-Furnished,Ground out of 1


In [14]:
Y_train

1196    2.597433
2215    2.341108
1598    2.241991
1532    2.379378
1501    2.323411
          ...   
4426    2.409345
466     2.389087
3092    2.389087
3772    2.513690
860     2.409345
Name: Rent, Length: 3559, dtype: float64

In [15]:
#Preprocessing
#Define the transformers
preprocessor = ColumnTransformer(
    transformers=[
        # (Name, Transformer, Columns to apply it to)
        ('num', StandardScaler(), numeric_data),           # Standardization for numbers
        ('cat', OneHotEncoder(handle_unknown='ignore'), category_data)    # One-Hot Encoding for text
    ]
)

In [18]:
# Building the 'conveyor belt' with Random Forest
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),      # Step 1: Clean and scale the data
                                        #preprocessor: 
                                        #This is your gatekeeper. It ensures that whenever you call model.predict(), the raw data is cleaned and 
                                        #transformed exactly the same way it was during training.
    
    ('regressor', RandomForestRegressor  
     (
        n_estimators=200,                # Number of trees(nodes)
                                         # n_estimators=200: You’re telling the model to grow 200 individual decision trees. Since this is a
                                        #Random Forest, it will average their results to give you a more stable prediction than a single tree could.
        random_state=42, 
        #max_depth=10                     # Prevents the trees from getting too complex
         n_jobs=-1                       #n_jobs=-1: This is the "turbo" button. It tells your computer to use all available CPU cores 
                                         #to build those 200 trees in parallel.  
    ))                                  
])

In [19]:
# Training the model
model_pipeline.fit(X_train, Y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
#predict
Y_pred = model_pipeline.predict(X_test)

In [21]:
#convert back from log scale
Y_test_actual = np.expm1(Y_test)
Y_pred_actual = np.expm1(Y_pred)

In [23]:
#CONVERT BACK FROM LOG SCALE
Y_test_actual = np.expm1(Y_test)
Y_pred_actual = np.expm1(Y_pred)

In [24]:
# Calculate evaluation metrics
mae = mean_absolute_error(Y_test_actual, Y_pred_actual)
mse = np.sqrt(mean_squared_error(Y_test_actual, Y_pred_actual))
r2 = r2_score(Y_test_actual, Y_pred_actual)                          # 2. How much of the variance did we capture? (0.0 to 1.0)

# Print the results
print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE): ",mse)
print("R2 Score:",r2)

Mean Absolute Error (MAE): 0.2898097362362757
Mean Squared Error (MSE):  0.38415424451711166
R2 Score: 0.8346967382645205
